In [26]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
from pathlib import Path

print("All common libraries imported successfully!")

All common libraries imported successfully!


In [6]:
from pathlib import Path

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

MODEL_DIR = BASE_DIR / "models"
KB_DIR = BASE_DIR / "knowledge_base"
DOC_DIR = KB_DIR / "documents"
VECTOR_DIR = KB_DIR / "vector_store"
RESULT_DIR = BASE_DIR / "results"

MODEL_PATH = MODEL_DIR / "it_helpdesk_classifier.pkl"
DB_PATH = BASE_DIR / "helpdesk.db"

for folder in [
    MODEL_DIR, KB_DIR, DOC_DIR,
    VECTOR_DIR, RESULT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("ALL PROJECT PATHS READY!")


ALL PROJECT PATHS READY!


In [ ]:
# Project configuration

In [1]:
from pathlib import Path

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

MODEL_DIR = BASE_DIR / "models"
KB_DIR = BASE_DIR / "knowledge_base"
VECTOR_DIR = KB_DIR / "vector_store"
RESULT_DIR = BASE_DIR / "results"

MODEL_PATH = MODEL_DIR / "it_helpdesk_classifier.pkl"

for folder in [MODEL_DIR, KB_DIR, VECTOR_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", BASE_DIR)
print("ML Model:", MODEL_PATH)
print("Vector Database:", VECTOR_DIR)

assert MODEL_PATH.exists(), "ML model file not found!"
assert VECTOR_DIR.exists(), "Vector database not found!"

print("\nProject configuration successful!")


Project: D:\AI_IT_HELPDESK
ML Model: D:\AI_IT_HELPDESK\models\it_helpdesk_classifier.pkl
Vector Database: D:\AI_IT_HELPDESK\knowledge_base\vector_store

Project configuration successful!


In [ ]:
# Import required packages

In [2]:
import joblib
import chromadb
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

print("Chatbot libraries imported successfully!")


Chatbot libraries imported successfully!


In [ ]:
# Load trained classifier

In [3]:
classifier = joblib.load(MODEL_PATH)

print("ML classifier loaded successfully!")

print("Model type:", type(classifier))
print("Available categories:")

for category in classifier.classes_:
    print("-", category)


ML classifier loaded successfully!
Model type: <class 'sklearn.pipeline.Pipeline'>
Available categories:
- Account & Password
- Network & VPN
- Printer
- Software
- System Health


In [ ]:
# Connect vector database

In [4]:
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DIR)
)

collection = chroma_client.get_collection(
    name="it_helpdesk_knowledge"
)

print("ChromaDB connected successfully!")
print("Collection:", collection.name)
print("Stored vectors:", collection.count())

assert collection.count() > 0, "Knowledge base is empty!"

ChromaDB connected successfully!
Collection: it_helpdesk_knowledge
Stored vectors: 15


In [ ]:
# Load embedding model

In [5]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print("Embedding model loaded!")

print(
    "Embedding dimensions:",
    embedding_model.get_embedding_dimension()
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Embedding dimensions: 384


In [ ]:
# Issue classification

In [6]:
def classify_it_issue(user_query):

    query = user_query.strip()

    if not query:
        raise ValueError("Please enter an IT problem.")

    predicted_category = classifier.predict([query])[0]

    confidence = None

    if hasattr(classifier, "predict_proba"):
        probabilities = classifier.predict_proba([query])[0]
        confidence = float(np.max(probabilities))

    return {
        "category": predicted_category,
        "confidence": confidence
    }


# Test classification
test_query = "My office printer is not working"

classification = classify_it_issue(test_query)

print("User Query:", test_query)
print("Predicted Category:", classification["category"])
print("Confidence:", classification["confidence"])

User Query: My office printer is not working
Predicted Category: Printer
Confidence: 0.32430132630576475


In [ ]:
# Integrated retrieval function

In [7]:
def retrieve_category_knowledge(
    query,
    category,
    top_k=3
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # First search within predicted category
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=min(top_k, collection.count()),
        where={"category": category},
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    # If category has no stored articles,
    # search the entire knowledge base
    if not results["ids"][0]:

        results = collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=min(top_k, collection.count()),
            include=[
                "documents",
                "metadatas",
                "distances"
            ]
        )

    retrieved = []

    for i in range(len(results["ids"][0])):

        retrieved.append({
            "content": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": float(results["distances"][0][i])
        })

    return retrieved


print("Category-aware retrieval function ready!")


Category-aware retrieval function ready!


In [ ]:
# Main chatbot engine

In [8]:
def ai_helpdesk(user_query):

    query = user_query.strip()

    if not query:
        return {
            "status": "INVALID",
            "answer": "Please enter your IT problem.",
            "sources": []
        }

    # STEP 1: Classification
    classification = classify_it_issue(query)

    category = classification["category"]
    confidence = classification["confidence"]

    # STEP 2: Low-confidence classification check
    if confidence is not None and confidence < 0.50:

        return {
            "status": "ESCALATION_REQUIRED",
            "category": category,
            "confidence": confidence,
            "answer": (
                "I could not confidently identify your IT issue. "
                "Please provide more details or contact an IT analyst."
            ),
            "sources": []
        }

    # STEP 3: Retrieve knowledge
    results = retrieve_category_knowledge(
        query,
        category,
        top_k=3
    )

    # STEP 4: Evidence validation
    if not results or results[0]["distance"] > 0.65:

        return {
            "status": "ESCALATION_REQUIRED",
            "category": category,
            "confidence": confidence,
            "answer": (
                "I could not find sufficiently relevant "
                "troubleshooting information. "
                "Your issue should be escalated to an IT analyst."
            ),
            "sources": []
        }

    # STEP 5: Build evidence-based response
    best_result = results[0]

    answer = (
        f"Your issue has been classified as: {category}\n\n"
        f"Recommended troubleshooting:\n\n"
        f"{best_result['content']}\n\n"
        "Please follow only authorised procedures. "
        "If the problem continues, contact IT support."
    )

    sources = []

    for result in results:

        metadata = result["metadata"]

        sources.append({
            "article_id": metadata["article_id"],
            "title": metadata["title"],
            "category": metadata["category"],
            "distance": round(result["distance"], 4)
        })

    return {
        "status": "ANSWER_GENERATED",
        "category": category,
        "confidence": confidence,
        "answer": answer,
        "sources": sources
    }


print("Integrated AI Helpdesk engine created!")

Integrated AI Helpdesk engine created!


In [ ]:
# Chatbot testing

In [9]:
test_queries = [
    "My office printer is offline",
    "I cannot connect to company VPN",
    "I forgot my Windows password",
    "My computer has high RAM usage",
    "My application keeps crashing",
    "My internet connection is not working"
]

for query in test_queries:

    result = ai_helpdesk(query)

    print("\n" + "=" * 65)
    print("USER:", query)
    print("=" * 65)

    print("STATUS:", result["status"])
    print("CATEGORY:", result.get("category", "Not classified"))
    print("CONFIDENCE:", result.get("confidence"))

    print("\nAI RESPONSE:")
    print(result["answer"])

    print("\nSOURCE REFERENCES:")

    for source in result["sources"]:
        print(
            source["article_id"],
            "-",
            source["title"]
        )


USER: My office printer is offline
STATUS: ESCALATION_REQUIRED
CATEGORY: Printer
CONFIDENCE: 0.35669720129177

AI RESPONSE:
I could not confidently identify your IT issue. Please provide more details or contact an IT analyst.

SOURCE REFERENCES:

USER: I cannot connect to company VPN
STATUS: ESCALATION_REQUIRED
CATEGORY: Network & VPN
CONFIDENCE: 0.34357663389466075

AI RESPONSE:
I could not confidently identify your IT issue. Please provide more details or contact an IT analyst.

SOURCE REFERENCES:

USER: I forgot my Windows password
STATUS: ESCALATION_REQUIRED
CATEGORY: Account & Password
CONFIDENCE: 0.4100217643517811

AI RESPONSE:
I could not confidently identify your IT issue. Please provide more details or contact an IT analyst.

SOURCE REFERENCES:

USER: My computer has high RAM usage
STATUS: ESCALATION_REQUIRED
CATEGORY: System Health
CONFIDENCE: 0.44851634158677717

AI RESPONSE:
I could not confidently identify your IT issue. Please provide more details or contact an IT analy

In [ ]:
# Install UI library

In [10]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Chatbot user interface

In [2]:
import ipywidgets as widgets

from IPython.display import display, clear_output

# User input
user_input = widgets.Textarea(
    value="",
    placeholder="Describe your IT problem here...",
    description="Problem:",
    layout=widgets.Layout(width="100%", height="100px")
)

# Submit button
submit_button = widgets.Button(
    description="Ask AI Helpdesk",
    button_style="primary",
    icon="comments"
)

# Output area
chat_output = widgets.Output(
    layout=widgets.Layout(
        width="100%",
        border="1px solid gray",
        padding="10px"
    )
)


def submit_query(button):

    with chat_output:

        clear_output(wait=True)

        query = user_input.value.strip()

        if not query:
            print("Please enter an IT problem.")
            return

        result = ai_helpdesk(query)

        display(Markdown("### AI IT HELPDESK"))

        print("USER:", query)
        print("\nSTATUS:", result["status"])
        print("CATEGORY:", result.get("category", "Unknown"))

        if result.get("confidence") is not None:
            print(
                "CLASSIFICATION CONFIDENCE:",
                round(result["confidence"] * 100, 2),
                "%"
            )

        display(Markdown("### Troubleshooting Response"))
        print(result["answer"])

        display(Markdown("### Knowledge References"))

        if result["sources"]:
            for source in result["sources"]:
                print(
                    source["article_id"],
                    "-",
                    source["title"]
                )
        else:
            print("No supporting knowledge article found.")

        display(Markdown("### User Confirmation"))

        print("Please verify whether the issue is resolved.")
        

submit_button.on_click(submit_query)

display(
    widgets.VBox([
        widgets.HTML(
            "<h2>AI-Based IT Helpdesk</h2>"
            "<p>Describe your technical problem to receive "
            "troubleshooting guidance.</p>"
        ),
        user_input,
        submit_button,
        chat_output
    ])
)


In [ ]:
# Feedback and escalation database

In [4]:
from pathlib import Path

# Main project directory
BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

# Project folders
MODEL_DIR = BASE_DIR / "models"
KB_DIR = BASE_DIR / "knowledge_base"
VECTOR_DIR = KB_DIR / "vector_store"
RESULT_DIR = BASE_DIR / "results"

# Database path
DB_PATH = BASE_DIR / "helpdesk.db"

# Create directories
for folder in [
    MODEL_DIR,
    KB_DIR,
    VECTOR_DIR,
    RESULT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DB_PATH:", DB_PATH)
print("Configuration successful!")


BASE_DIR: D:\AI_IT_HELPDESK
DB_PATH: D:\AI_IT_HELPDESK\helpdesk.db
Configuration successful!


In [5]:
import sqlite3
from datetime import datetime

DB_PATH = BASE_DIR / "helpdesk.db"

connection = sqlite3.connect(DB_PATH)

connection.execute("""
CREATE TABLE IF NOT EXISTS chatbot_feedback (

    id INTEGER PRIMARY KEY AUTOINCREMENT,
    created_at TEXT,
    user_query TEXT,
    predicted_category TEXT,
    confidence REAL,
    status TEXT,
    resolution_status TEXT,
    feedback TEXT

)
""")

connection.commit()
connection.close()

print("Chatbot feedback table created successfully!")
print("Database location:", DB_PATH)

Chatbot feedback table created successfully!
Database location: D:\AI_IT_HELPDESK\helpdesk.db


In [9]:
def ai_helpdesk(user_query):

    query = user_query.strip()

    if not query:
        return {
            "status": "INVALID",
            "answer": "Please enter your IT problem.",
            "sources": []
        }

    # Step 1: Predict category
    classification = classify_it_issue(query)

    category = classification["category"]
    confidence = classification["confidence"]

    # Step 2: Check confidence
    if confidence is not None and confidence < 0.50:

        return {
            "status": "ESCALATION_REQUIRED",
            "category": category,
            "confidence": confidence,
            "answer": "Unable to confidently classify the issue. Please contact IT support.",
            "sources": []
        }

    # Step 3: Retrieve relevant knowledge
    results = retrieve_category_knowledge(
        query,
        category,
        top_k=3
    )

    # Step 4: Check retrieval evidence
    if not results or results[0]["distance"] > 0.65:

        return {
            "status": "ESCALATION_REQUIRED",
            "category": category,
            "confidence": confidence,
            "answer": "No sufficiently relevant knowledge found. Please escalate to an IT analyst.",
            "sources": []
        }

    # Step 5: Generate response
    best_result = results[0]

    answer = (
        f"Your issue category: {category}\n\n"
        f"Troubleshooting guidance:\n\n"
        f"{best_result['content']}\n\n"
        "If the issue is not resolved, contact IT support."
    )

    sources = []

    for result in results:

        metadata = result["metadata"]

        sources.append({
            "article_id": metadata["article_id"],
            "title": metadata["title"],
            "category": metadata["category"],
            "distance": round(result["distance"], 4)
        })

    return {
        "status": "ANSWER_GENERATED",
        "category": category,
        "confidence": confidence,
        "answer": answer,
        "sources": sources
    }


print("AI Helpdesk function created successfully!")


AI Helpdesk function created successfully!


In [11]:
import numpy as np

def classify_it_issue(user_query):

    query = user_query.strip()

    if not query:
        raise ValueError("Please enter an IT problem.")

    # Predict IT issue category
    predicted_category = classifier.predict([query])[0]

    confidence = None

    # Calculate prediction confidence
    if hasattr(classifier, "predict_proba"):

        probabilities = classifier.predict_proba([query])[0]

        confidence = float(np.max(probabilities))

    return {
        "category": predicted_category,
        "confidence": confidence
    }


print("Issue classification function created successfully!")

Issue classification function created successfully!


In [13]:
from pathlib import Path
import joblib

# Project path
BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

# Model file path
MODEL_PATH = (
    BASE_DIR / "models" /
    "it_helpdesk_classifier.pkl"
)

# Check model file
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}"
    )

# Load trained model
classifier = joblib.load(MODEL_PATH)

print("AI Classification Model Loaded!")
print("Model:", type(classifier))
print("Available categories:")

for category in classifier.classes_:
    print("-", category)


AI Classification Model Loaded!
Model: <class 'sklearn.pipeline.Pipeline'>
Available categories:
- Account & Password
- Network & VPN
- Printer
- Software
- System Health


In [14]:
sample_query = "My printer is offline"

sample_result = ai_helpdesk(sample_query)

save_chatbot_feedback(
    sample_query,
    sample_result,
    resolution_status="Pending",
    feedback="Initial chatbot testing"
)

Interaction saved successfully!


In [15]:
test_query = "My printer is offline"

prediction = classifier.predict([test_query])

print("User Query:", test_query)
print("Predicted Category:", prediction[0])

if hasattr(classifier, "predict_proba"):
    probabilities = classifier.predict_proba([test_query])[0]
    print("Confidence:", round(float(max(probabilities)) * 100, 2), "%")


User Query: My printer is offline
Predicted Category: Printer
Confidence: 43.54 %


In [16]:
sample_query = "My printer is offline"

sample_result = ai_helpdesk(sample_query)

print("Status:", sample_result["status"])
print("Category:", sample_result.get("category"))
print("AI Response:")
print(sample_result["answer"])

save_chatbot_feedback(
    sample_query,
    sample_result,
    resolution_status="Pending",
    feedback="Initial chatbot testing"
)


Status: ESCALATION_REQUIRED
Category: Printer
AI Response:
Unable to confidently classify the issue. Please contact IT support.
Interaction saved successfully!


In [ ]:
# Unknown issue test

In [17]:

unknown_query = (
    "My specialised business application "
    "has an unusual database synchronisation error"
)

unknown_result = ai_helpdesk(unknown_query)

print("USER:", unknown_query)
print("\nSTATUS:", unknown_result["status"])
print("\nRESPONSE:")
print(unknown_result["answer"])

USER: My specialised business application has an unusual database synchronisation error

STATUS: ESCALATION_REQUIRED

RESPONSE:
Unable to confidently classify the issue. Please contact IT support.


In [ ]:
# Save chatbot test results

In [19]:
from pathlib import Path
import joblib

# Project path
BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

# Model file path
MODEL_PATH = (
    BASE_DIR / "models" /
    "it_helpdesk_classifier.pkl"
)

# Check model file
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found: {MODEL_PATH}"
    )

# Load trained model
classifier = joblib.load(MODEL_PATH)

print("AI Classification Model Loaded!")
print("Model:", type(classifier))
print("Available categories:")

for category in classifier.classes_:
    print("-", category)


AI Classification Model Loaded!
Model: <class 'sklearn.pipeline.Pipeline'>
Available categories:
- Account & Password
- Network & VPN
- Printer
- Software
- System Health


In [22]:
test_queries = [
    "My printer is offline",
    "I cannot connect to the company VPN",
    "I forgot my Windows password",
    "My computer is running very slowly",
    "Microsoft Outlook is not opening",
    "I cannot access my user account",
    "The network connection is not working",
    "My printer is showing a paper jam error",
    "I need help installing software",
    "My computer is overheating"
]

print("Total test queries:", len(test_queries))

for i, query in enumerate(test_queries, start=1):
    print(f"{i}. {query}")


Total test queries: 10
1. My printer is offline
2. I cannot connect to the company VPN
3. I forgot my Windows password
4. My computer is running very slowly
5. Microsoft Outlook is not opening
6. I cannot access my user account
7. The network connection is not working
8. My printer is showing a paper jam error
9. I need help installing software
10. My computer is overheating


In [24]:
import pandas as pd

print("Pandas imported successfully!")
print("Pandas version:", pd.__version__)


chatbot_test_results = []

for query in test_queries:

    result = ai_helpdesk(query)

    chatbot_test_results.append({
        "User Query": query,
        "Predicted Category": result.get("category"),
        "Confidence": result.get("confidence"),
        "Response": result.get("response"),
        "Status": result.get("status")
    })

chatbot_test_df = pd.DataFrame(chatbot_test_results)

display(chatbot_test_df)

Pandas imported successfully!
Pandas version: 3.0.3


,User Query,Predicted Category,Confidence,Response,Status
0,My printer is offline,Printer,0.435424,None,ESCALATION_REQUIRED
1,I cannot connect to the company VPN,Network & VPN,0.363764,None,ESCALATION_REQUIRED
2,I forgot my Windows password,Account & Password,0.410022,None,ESCALATION_REQUIRED
3,My computer is running very slowly,System Health,0.433452,None,ESCALATION_REQUIRED
4,Microsoft Outlook is not opening,Software,0.310836,None,ESCALATION_REQUIRED
5,I cannot access my user account,Account & Password,0.266023,None,ESCALATION_REQUIRED
6,The network connection is not working,Network & VPN,0.366123,None,ESCALATION_REQUIRED
7,My printer is showing a paper jam error,Printer,0.440710,None,ESCALATION_REQUIRED
8,I need help installing software,Software,0.364538,None,ESCALATION_REQUIRED
9,My computer is overheating,System Health,0.437359,None,ESCALATION_REQUIRED


In [25]:
chatbot_test_df = pd.DataFrame(chatbot_test_results)

display(chatbot_test_df)


,User Query,Predicted Category,Confidence,Response,Status
0,My printer is offline,Printer,0.435424,None,ESCALATION_REQUIRED
1,I cannot connect to the company VPN,Network & VPN,0.363764,None,ESCALATION_REQUIRED
2,I forgot my Windows password,Account & Password,0.410022,None,ESCALATION_REQUIRED
3,My computer is running very slowly,System Health,0.433452,None,ESCALATION_REQUIRED
4,Microsoft Outlook is not opening,Software,0.310836,None,ESCALATION_REQUIRED
5,I cannot access my user account,Account & Password,0.266023,None,ESCALATION_REQUIRED
6,The network connection is not working,Network & VPN,0.366123,None,ESCALATION_REQUIRED
7,My printer is showing a paper jam error,Printer,0.440710,None,ESCALATION_REQUIRED
8,I need help installing software,Software,0.364538,None,ESCALATION_REQUIRED
9,My computer is overheating,System Health,0.437359,None,ESCALATION_REQUIRED


In [21]:
chatbot_test_results = []

for query in test_queries:

    result = ai_helpdesk(query)

    chatbot_test_results.append({
        "User_Query": query,
        "Predicted_Category": result.get("category"),
        "Confidence": result.get("confidence"),
        "Status": result["status"],
        "Sources_Found": len(result["sources"])
    })

chatbot_df = pd.DataFrame(chatbot_test_results)

CHATBOT_RESULTS = (
    RESULT_DIR / "chatbot_integration_results.csv"
)

chatbot_df.to_csv(
    CHATBOT_RESULTS,
    index=False,
    encoding="utf-8-sig"
)

display(chatbot_df)

print("\nChatbot evaluation exported!")
print("Location:", CHATBOT_RESULTS)

Total test queries: 10
1. My printer is offline
2. I cannot connect to the company VPN
3. I forgot my Windows password
4. My computer is running very slowly
5. Microsoft Outlook is not opening
6. I cannot access my user account
7. The network connection is not working
8. My printer is showing a paper jam error
9. I need help installing software
10. My computer is overheating


In [27]:
chatbot_test_results = []

for query in test_queries:

    result = ai_helpdesk(query)

    chatbot_test_results.append({
        "User_Query": query,
        "Predicted_Category": result.get("category"),
        "Confidence": result.get("confidence"),
        "Status": result["status"],
        "Sources_Found": len(result["sources"])
    })

chatbot_df = pd.DataFrame(chatbot_test_results)

CHATBOT_RESULTS = (
    RESULT_DIR / "chatbot_integration_results.csv"
)

In [ ]:
# Check SQLite database

In [28]:

with sqlite3.connect(DB_PATH) as conn:

    feedback_df = pd.read_sql_query(
        "SELECT * FROM chatbot_feedback",
        conn
    )

display(feedback_df)

print("Total recorded interactions:", len(feedback_df))

,id,created_at,user_query,predicted_category,confidence,status,resolution_status,feedback
0,1,2026-11-02T13:55:24,My printer is offline,Printer,0.435424,ESCALATION_REQUIRED,Pending,Initial chatbot testing
1,2,2026-11-02T13:56:17,My printer is offline,Printer,0.435424,ESCALATION_REQUIRED,Pending,Initial chatbot testing


Total recorded interactions: 2


In [29]:
from pathlib import Path

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")
FRONTEND_DIR = BASE_DIR / "frontend"
FRONTEND_DIR.mkdir(parents=True, exist_ok=True)

APP_PATH = FRONTEND_DIR / "app.py"

print("Application folder:", FRONTEND_DIR)
print("App file path:", APP_PATH)

Application folder: D:\AI_IT_HELPDESK\frontend
App file path: D:\AI_IT_HELPDESK\frontend\app.py


In [ ]:
# Create Streamlit application

In [30]:
app_code = r'''
import streamlit as st
from pathlib import Path
import joblib
import chromadb
from sentence_transformers import SentenceTransformer

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")

MODEL_PATH = BASE_DIR / "models" / "it_helpdesk_classifier.pkl"
VECTOR_DIR = BASE_DIR / "knowledge_base" / "vector_store"

st.set_page_config(
    page_title="AI IT Helpdesk",
    page_icon="🤖",
    layout="wide"
)

@st.cache_resource
def load_ai_components():
    classifier = joblib.load(MODEL_PATH)

    client = chromadb.PersistentClient(
        path=str(VECTOR_DIR)
    )

    collection = client.get_collection(
        name="it_helpdesk_knowledge"
    )

    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu"
    )

    return classifier, collection, embedding_model

classifier, collection, embedding_model = load_ai_components()

def classify_issue(query):
    category = classifier.predict([query])[0]

    confidence = None
    if hasattr(classifier, "predict_proba"):
        probabilities = classifier.predict_proba([query])[0]
        confidence = float(max(probabilities))

    return category, confidence

def retrieve_knowledge(query, category, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where={"category": category}
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    return documents, metadatas

st.title("🤖 AI-Based IT Helpdesk")
st.caption("AI Issue Classification and Knowledge-Based Troubleshooting")

st.sidebar.title("Helpdesk Menu")
st.sidebar.info("AI-powered IT support assistant")

st.subheader("How can we help you?")

user_query = st.text_area(
    "Describe your IT issue:",
    placeholder="Example: My printer is offline and I cannot print documents...",
    height=120
)

if st.button("Submit Issue", type="primary"):

    if not user_query.strip():
        st.warning("Please enter your IT issue.")

    else:
        with st.spinner("AI is analysing your issue..."):

            category, confidence = classify_issue(user_query)

            documents, metadatas = retrieve_knowledge(
                user_query,
                category
            )

        st.success("Issue analysis completed!")

        st.subheader("AI Classification")

        col1, col2 = st.columns(2)

        with col1:
            st.metric("Predicted Category", str(category))

        with col2:
            if confidence is not None:
                st.metric(
                    "Classification Confidence",
                    f"{confidence:.2%}"
                )

        st.subheader("Troubleshooting Guidance")

        if documents:
            for i, document in enumerate(documents):
                title = metadatas[i].get(
                    "title", f"Knowledge Article {i+1}"
                )

                with st.expander(title, expanded=(i == 0)):
                    st.write(document)

        else:
            st.warning(
                "No matching knowledge article was found. "
                "Please contact the IT support team."
            )

        st.subheader("Support Status")

        if confidence is not None and confidence < 0.50:
            st.warning(
                "Low classification confidence. "
                "Manual IT support review is recommended."
            )
        elif not documents:
            st.warning(
                "Manual IT support review is recommended."
            )
        else:
            st.info(
                "Suggested troubleshooting guidance is available. "
                "Follow the instructions and contact IT support "
                "if the issue remains unresolved."
            )

st.divider()

st.caption(
    "MSc Computer Science Project | AI-Based IT Helpdesk "
    "and Automated Troubleshooting System"
)
'''

APP_PATH.write_text(app_code, encoding="utf-8")

print("Streamlit application created successfully!")
print(APP_PATH)

Streamlit application created successfully!
D:\AI_IT_HELPDESK\frontend\app.py


In [31]:

import subprocess
import sys

subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_PATH)
])

print("Streamlit application is starting...")
print("Open: http://localhost:8501")


Streamlit application is starting...
Open: http://localhost:8501


In [32]:
import sys

print("Python Executable:")
print(sys.executable)

print("\nPython Version:")
print(sys.version)


Python Executable:
C:\Users\Nimesh\anaconda3\python.exe

Python Version:
3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]


In [33]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "streamlit"],
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else "Streamlit is not installed.")

Name: streamlit
Version: 1.58.0
Summary: A faster way to build and share data apps
Home-page: https://streamlit.io
Author: 
Author-email: Snowflake Inc <hello@streamlit.io>
License-Expression: Apache-2.0
Location: C:\Users\Nimesh\anaconda3\Lib\site-packages
Requires: altair, anyio, blinker, cachetools, click, gitpython, httptools, itsdangerous, numpy, packaging, pandas, pillow, protobuf, pyarrow, python-multipart, requests, starlette, tenacity, toml, typing-extensions, uvicorn, watchdog, websockets
Required-by: 



In [34]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [1]:
import subprocess
import sys
from pathlib import Path

APP_PATH = Path(r"D:\AI_IT_HELPDESK\frontend\app.py")

if not APP_PATH.exists():
    raise FileNotFoundError(f"App file not found: {APP_PATH}")

process = subprocess.Popen([
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_PATH)
])

print("Streamlit process started.")
print("Process ID:", process.pid)
print("Open: http://localhost:8501")

Streamlit process started.
Process ID: 9256
Open: http://localhost:8501


In [2]:
import sys
import subprocess
import time
from pathlib import Path

BASE_DIR = Path(r"D:\AI_IT_HELPDESK")
APP_PATH = BASE_DIR / "frontend" / "app.py"
LOG_DIR = BASE_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "streamlit.log"

# Check application file
if not APP_PATH.exists():
    raise FileNotFoundError(f"App file not found: {APP_PATH}")

# Check Streamlit installation
check = subprocess.run(
    [sys.executable, "-m", "streamlit", "--version"],
    capture_output=True,
    text=True
)

if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError("Streamlit is not installed in this Python environment.")

print(check.stdout)

# Start Streamlit and save errors to log
log_file = open(LOG_PATH, "w", encoding="utf-8")

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_PATH),
        "--server.port=8501",
        "--server.headless=true"
    ],
    cwd=str(APP_PATH.parent),
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

if process.poll() is None:
    print("SUCCESS: Streamlit process is running.")
    print("Process ID:", process.pid)
    print("Open: http://localhost:8501")
else:
    log_file.close()
    print("ERROR: Streamlit stopped during startup.")
    print("\nStartup log:")
    print(LOG_PATH.read_text(encoding="utf-8", errors="replace"))


Streamlit, version 1.58.0

SUCCESS: Streamlit process is running.
Process ID: 24500
Open: http://localhost:8501


In [4]:
import pandas as pd
import sqlite3
from pathlib import Path

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [5]:
import sqlite3
from pathlib import Path

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name
    """,
    conn
)

display(tables)

conn.close()


,name
0,audit_logs
1,categories
2,conversations
3,knowledge_base
4,sqlite_sequence
5,tickets
6,users


In [6]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path(r"D:\AI_IT_HELPDESK\database\helpdesk.db")

conn = sqlite3.connect(DB_PATH)

for table_name in ["tickets", "users", "categories", "conversations", "audit_logs"]:
    print("\n" + "=" * 60)
    print("TABLE:", table_name)
    print("=" * 60)

    schema = pd.read_sql_query(
        f"PRAGMA table_info({table_name})",
        conn
    )

    display(schema)

conn.close()



TABLE: tickets


,cid,name,type,notnull,dflt_value,pk
0,0,ticket_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,title,TEXT,1,NaN,0
3,3,description,TEXT,0,NaN,0
4,4,category_id,INTEGER,0,NaN,0
5,5,priority,TEXT,0,'Medium',0
6,6,status,TEXT,0,'Open',0
7,7,created_at,DATETIME,0,CURRENT_TIMESTAMP,0
8,8,updated_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: users


,cid,name,type,notnull,dflt_value,pk
0,0,user_id,INTEGER,0,NaN,1
1,1,username,TEXT,1,NaN,0
2,2,email,TEXT,0,NaN,0
3,3,password_hash,TEXT,0,NaN,0
4,4,role,TEXT,0,'user',0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: categories


,cid,name,type,notnull,dflt_value,pk
0,0,category_id,INTEGER,0,None,1
1,1,category_name,TEXT,1,None,0
2,2,description,TEXT,0,None,0



TABLE: conversations


,cid,name,type,notnull,dflt_value,pk
0,0,conversation_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,ticket_id,INTEGER,0,NaN,0
3,3,user_message,TEXT,0,NaN,0
4,4,ai_response,TEXT,0,NaN,0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0



TABLE: audit_logs


,cid,name,type,notnull,dflt_value,pk
0,0,log_id,INTEGER,0,NaN,1
1,1,user_id,INTEGER,0,NaN,0
2,2,ticket_id,INTEGER,0,NaN,0
3,3,action,TEXT,1,NaN,0
4,4,details,TEXT,0,NaN,0
5,5,created_at,DATETIME,0,CURRENT_TIMESTAMP,0
